# Smart Pet Vision — Multiple Animals v0.4: Real-weight validation

**This notebook is NOT an executed experiment.** It runs a frozen, paired **401-image regression** comparing the same model's baseline and candidate safety decisions. It does not train, edit historical results, or establish independent-holdout accuracy. Staff approval is mandatory for all outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q 'ultralytics==8.4.182' 'pandas' 'pillow' 'numpy' 'fastapi' 'httpx'


## Configuration (change only these paths)

Add a Drive shortcut to the project folder if it is shared with you. Copy the candidate ZIP into your Drive. **Do not paste credentials here.**

In [ ]:
from pathlib import Path
# Replace with actual authorized Drive paths on the Colab machine.
ARCHIVE = Path('/content/drive/MyDrive/SmartPetVision_Phase2_MultiAnimal_Source_v0.4.zip')
DATA_IMAGES = Path('/content/drive/MyDrive/SmartPetVision_TeamData/images')
DATA_LABELS = Path('/content/drive/MyDrive/SmartPetVision_TeamData/labels.csv')
WEIGHTS_DETECTOR = Path('/content/drive/MyDrive/capstone /smart-pet-vision/runs/detect/yolo11s_det/weights/best.pt')
WEIGHTS_SEGMENTER = Path('/content/drive/MyDrive/capstone /smart-pet-vision/runs/segment/yolo11s_seg/weights/best.pt')
WEIGHTS_CLASSIFIER = Path('/content/drive/MyDrive/capstone /smart-pet-vision/runs/classification/finetune.pt')
RESULTS_BASE = Path('/content/drive/MyDrive/SmartPetVision_Phase2_Review_Runs')
for path in (ARCHIVE,DATA_IMAGES,DATA_LABELS,WEIGHTS_DETECTOR,WEIGHTS_SEGMENTER,WEIGHTS_CLASSIFIER):
    if not path.exists(): raise FileNotFoundError(f'Required input unavailable: {path}')


## Freeze model source and run in a new directory only

In [ ]:
import json,hashlib,zipfile,subprocess,os,sys,time
WORK=Path('/content/spv_multiple_animals_v04')
if WORK.exists(): raise FileExistsError('Choose a fresh Colab runtime to avoid overwriting previous work.')
WORK.mkdir(parents=True)
with zipfile.ZipFile(ARCHIVE) as z: z.extractall(WORK)
manifest=json.loads((WORK/'runs/manifest_multi_animal_candidate.json').read_text())
manifest.update(detector=str(WEIGHTS_DETECTOR),segmenter=str(WEIGHTS_SEGMENTER),classifier=str(WEIGHTS_CLASSIFIER))
manifest['review_mode']='enforce'
RUNTIME_MANIFEST=WORK/'runs/manifest_multi_animal_runtime.json'
if RUNTIME_MANIFEST.exists():raise FileExistsError('Runtime manifest must be new.')
RUNTIME_MANIFEST.write_text(json.dumps(manifest,indent=2))
name=time.strftime('v04_401_regression_%Y%m%d_%H%M%S')
OUT=RESULTS_BASE/name
if OUT.exists():raise FileExistsError(f'Existing results: {OUT}')
RESULTS_BASE.mkdir(parents=True,exist_ok=True)
env=os.environ.copy(); env['PYTHONPATH']=str(WORK/'src')
cmd=[sys.executable,str(WORK/'tools/evaluate_multi_animal_candidate.py'),'--manifest',str(RUNTIME_MANIFEST),
     '--images',str(DATA_IMAGES),'--labels',str(DATA_LABELS),'--output',str(OUT)]
run=subprocess.run(cmd, cwd=str(WORK),env=env,text=True,check=True,capture_output=True)
print(run.stdout)
print(f'REAL RESULT DIRECTORY: {OUT}')


## Evidence review

Open `paired_summary.json` and `per_image_regression.csv`. Do **not** choose confidence or IoU thresholds on the 401 previously inspected photos. A new independent holdout and T4 latency benchmark are required before final approval.

In [ ]:
import json
summary=json.loads((OUT/'paired_summary.json').read_text())
print('Multi-animal provisional READY: before',summary['baseline']['multi_animal_ready'],
      'after',summary['candidate']['multi_animal_ready'])
print('Single-animal referrals: before',summary['baseline']['single_animal_referrals'],
      'after',summary['candidate']['single_animal_referrals'])
print('Holdout claim?',summary['independent_holdout'])
